# Financial Bank Statement Extraction - Generating Complex PDFs for Evaluations

This notebook evaluates how well OpenAI models can extract specific financial figures from bank financial statement PDFs.

**Workflow:**
1. Load PDF files from the `files/` directory
2. Ask a fixed set of questions to the OpenAI API (with PDF content)
3. Compare answers against golden labels
4. Upload the PDFs to DataFramer as seed files and generate a Specification

## Setup

In [ ]:
import os

!git clone https://github.com/aimonlabs/dataframer-docs-public.git
os.chdir("dataframer-docs-public/financial-bank-text-extraction-evaluation")

In [ ]:
!pip install openai pymupdf pandas pydataframer tenacity pyyaml requests

import io
import os
import zipfile
from datetime import datetime
from pathlib import Path

import fitz  # PyMuPDF
import pandas as pd
from openai import OpenAI

import dataframer
from dataframer import Dataframer
from tenacity import retry, retry_if_result, stop_never, wait_fixed

openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
df_client = Dataframer(api_key=os.environ["DATAFRAMER_API_KEY"])

FILES_DIR = Path("files")
MODEL = "gpt-4o"

print(f"Dataframer SDK version: {dataframer.__version__}")

## Questions & Golden Labels

Each question has a key used to look up the per-PDF golden (expected) answer.
Fill in `GOLDEN_LABELS` after manually verifying the correct values in each PDF.

In [ ]:
# Ordered list of (question_key, question_text)
QUESTIONS = [
    (
        "q1_cre_to_tier1_plus_acl_pct",
        "What is the bank's total commercial real estate loans as a percentage of Tier 1 Capital plus ACL "
        "compare to the regulatory supervisory threshold of 300%? "
        "Print only the bank's CRE-to-(Tier-1-plus-ACL) percentage as a single number.",
    ),
    (
        "q2_non_owner_occ_cre_3yr_growth",
        "Within the non-owner occupied CRE bucket, what is the bank's three-year growth rate "
        "relative to its peer group? "
        "Print only the bank's three-year growth rate as a single number or percentage.",
    ),
    (
        "q3_single_category_concentration",
        "What share of the bank's total loan portfolio is concentrated in a single loan category, "
        "Print only the percentage share of the largest single loan category.",
    ),
    (
        "q4_1_4_family_residential_to_tier1",
        "What is the bank's 1-4 family residential exposure as a percentage of Tier 1 Capital? "
        "Print only a single percentage.",
    ),
    (
        "q5_non_depository_growth_vs_tier1",
        "Is the bank's non-depository and other loans category growing faster than its Tier 1 Capital base, "
        "potentially signaling undisclosed concentration risk? Reply with N/A if 0.",
    ),
]

# Golden labels per PDF (keyed by PDF filename stem, then question key).
# Set values to None where the answer is unknown or not applicable for that document.
GOLDEN_LABELS: dict[str, dict[str, str | None]] = {
    "CATHAY BANK": {
        "q1_cre_to_tier1_plus_acl_pct": "344.49",         # Total CRE as % of Tier 1 + ACL at 12/31/2025; exceeds 300% supervisory threshold
        "q2_non_owner_occ_cre_3yr_growth": "18.77",        # NOO CRE 3-year growth rate at 12/31/2025; below peer avg of 23.57%
        "q3_single_category_concentration": "49.17",       # NOO CRE as % of total loans at 12/31/2025; just crosses 40% threshold
        "q4_1_4_family_residential_to_tier1": "245.59",    # 1-4 Family Residential as % of Tier 1 + ACL at 12/31/2025
        "q5_non_depository_growth_vs_tier1": "False",      # No sustained growth trend; net flat-to-down over 5-year window
    },
    "BANK OF AMERICA": {
        "q1_cre_to_tier1_plus_acl_pct": "0.00",              # Bank's Total CRE as % of Tier 1 + ACL at 12/31/2025
        "q2_non_owner_occ_cre_3yr_growth": "N/A",           # Bank reports N/A — zero NOO CRE balances in all periods
        "q3_single_category_concentration": "100.00",          # ~100% of gross loans are in 1-4 Family Residential
        "q4_1_4_family_residential_to_tier1": "512.59",       # Bank's 1-4 Family Residential as % of Tier 1 + ACL
        "q5_non_depository_growth_vs_tier1": "N/A",         # Bank reports 0.00 in Non-Depository & Other in all periods — no balance, no growth, no signal
    },
    "JP MORGAN": {
        "q1_cre_to_tier1_plus_acl_pct": "66.30",          # Total CRE as % of Tier 1 + ACL at 12/31/2025; well below 300% threshold
        "q2_non_owner_occ_cre_3yr_growth": "48.02",        # NOO CRE 3-year growth rate at 12/31/2025; well above peer avg of 9.98% (82nd pct)
        "q3_single_category_concentration": "28.27",       # No single category exceeds 40%; Total CRE is only 14.18% of total loans
        "q4_1_4_family_residential_to_tier1": "103.58",    # 1-4 Family Residential as % of Tier 1 + ACL at 12/31/2025
        "q5_non_depository_growth_vs_tier1": "True",       # Non-Depository & Other: 94.55→93.72→105.95→108.45→132.15; ~21.8% jump in 2025
    },
    "MORGAN STANLEY": {
        "q1_cre_to_tier1_plus_acl_pct": "43.90",          # Total CRE as % of Tier 1 + ACL at 12/31/2025; far below 300% threshold (32nd pct vs peers)
        "q2_non_owner_occ_cre_3yr_growth": "12.87",        # NOO CRE 3-year growth rate at 12/31/2025; modestly above peer avg of 9.98% (64th pct); sharply decelerating
        "q3_single_category_concentration": "53.91",        # Non-Depository & Other dominates: 532.56 / ~987.77 (sum of all lines as % of Tier 1) ≈ 53.9% of gross loans
        "q4_1_4_family_residential_to_tier1": "410.13",    # 1-4 Family Residential as % of Tier 1 + ACL at 12/31/2025 (96th pct; nearly 4x peer avg)
        "q5_non_depository_growth_vs_tier1": "True",       # Non-Depository & Other: 312.36→225.32→223.04→426.44→532.56; ~91% jump in 2024, ~25% in 2025
    },
}

## Load PDFs

In [ ]:
def extract_pdf_text(pdf_path: Path) -> str:
    doc = fitz.open(pdf_path)
    return "\n\n".join(page.get_text() for page in doc)


pdf_texts: dict[str, str] = {}
for pdf_path in sorted(FILES_DIR.glob("*.pdf")):
    text = extract_pdf_text(pdf_path)
    pdf_texts[pdf_path.stem] = text
    print(f"Loaded '{pdf_path.name}' — {len(text):,} chars")

## Query OpenAI

For each PDF, ask every question and record the model's answer alongside the golden label.

In [ ]:
SYSTEM_PROMPT = (
    "You are a precise financial data extraction assistant. "
    "The user will provide the full text of a bank financial statement and ask a specific question. "
    "Answer with only the single requested value — no explanation, no units label, no extra text."
    "Important - If answering with a number, answer with 2 decimal places."
)


def ask_question(pdf_text: str, question: str) -> str:
    response = openai_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"Financial statement:\n\n{pdf_text}\n\nQuestion: {question}",
            },
        ],
        temperature=0,
    )
    return response.choices[0].message.content.strip()


results = []

for file_stem, pdf_text in pdf_texts.items():
    print(f"\n--- {file_stem} ---")
    golden_for_file = GOLDEN_LABELS.get(file_stem, {})
    for q_key, q_text in QUESTIONS:
        answer = ask_question(pdf_text, q_text)
        golden = golden_for_file.get(q_key)
        match = (answer == golden) if golden is not None else None
        results.append({
            "file": file_stem,
            "question_key": q_key,
            "question": q_text,
            "model_answer": answer,
            "golden": golden,
            "exact_match": match,
        })
        status = "✓" if match is True else ("?" if match is None else "✗")
        print(f"  [{status}] {q_key}: {answer!r}  (golden: {golden!r})")

## Results

In [ ]:
df = pd.DataFrame(results)
pd.set_option("display.max_colwidth", 120)
df[["file", "question_key", "model_answer", "golden", "exact_match"]]

In [ ]:
labeled = df[df["golden"].notna()]
if labeled.empty:
    print("No golden labels filled in yet — populate GOLDEN_LABELS to see accuracy metrics.")
else:
    accuracy = labeled["exact_match"].mean()
    print(f"Overall exact-match accuracy: {accuracy:.1%} ({int(labeled['exact_match'].sum())}/{len(labeled)})")
    print()
    print("Per-question accuracy:")
    print(labeled.groupby("question_key")["exact_match"].mean().to_string())

---

## DataFramer SDK — Generate a Specification from the PDFs

We upload the bank statement PDFs as a seed dataset, then let DataFramer analyze them and produce a [Specification](https://docs.dataframer.ai/essentials/specifications) that captures the structure and patterns of the documents. This spec can later be used to generate synthetic bank statement datasets at scale.

### Step 1: Upload PDF Files as a Seed Dataset

All PDFs from the `files/` directory are packed into a ZIP in memory and uploaded to DataFramer.

In [ ]:
pdf_paths = sorted(FILES_DIR.glob("*.pdf"))

zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    for pdf_path in pdf_paths:
        zf.write(pdf_path, arcname=pdf_path.name)
        print(f"  Added: {pdf_path.name}")
zip_buffer.seek(0)

dataset = df_client.dataframer.seed_datasets.create_from_zip(
    name=f"bank_analysis_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    description="Bank Analysis - Concentrations of Credit PDFs for extraction evaluation",
    zip_file=zip_buffer,
)

dataset_id = dataset.id
print(f"\nUpload complete")
print(f"Dataset ID: {dataset_id}")
print(f"Files: {dataset.file_count}")

### Step 2: Generate a Specification

DataFramer analyzes the seed PDFs and produces a spec that describes the document structure, data properties, and value distributions.

In [ ]:
spec_name = f"with_opus_bank_statements_spec_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
spec = df_client.dataframer.specs.create(
    dataset_id=dataset_id,
    name=spec_name,
    spec_generation_model_name="anthropic/claude-opus-4-7", # Check out available models on docs.dataframer.ai
    generation_objectives=
        "Do include " + 
        "'Real Estate Loans > Construction & Development > 1-4 Family Construction concentration level (BANK value at 12/31/2025)'" + 
        " and" +
        "'Real Estate Loans > Multifamily concentration level (BANK value at 12/31/2025) ' as data properties. " +
        "Do not include HTML or CSS related data properties",
    ## Let's check out the distirbutions of data values as is in the data first
    extrapolate_values=False,
    generate_distributions=True,
)

spec_id = spec.id
print(f"Started spec analysis: {spec_name}")
print(f"Spec ID: {spec_id}")


def spec_not_ready(result):
    return result.status not in ("SUCCEEDED", "FAILED")


@retry(wait=wait_fixed(5), retry=retry_if_result(spec_not_ready), stop=stop_never)
def poll_spec_status(spec_id):
    return df_client.dataframer.specs.retrieve(spec_id=spec_id)


print("Polling for spec status...")
spec_result = poll_spec_status(spec_id)

if spec_result.status == "FAILED":
    raise RuntimeError("Spec analysis failed")

print(f"\nSpec analysis completed successfully!")
print(f"Spec ID: {spec_id}")

### Step 3: Review the Generated Specification

In [ ]:
import yaml

spec = df_client.dataframer.specs.retrieve(spec_id=spec_id)
config = yaml.safe_load(spec.content_yaml)
spec_data = config.get("spec", config)

print(f"Spec YAML length: {len(spec.content_yaml):,} chars")
print()

if "data_property_variations" in spec_data:
    print("Data properties:")
    for prop in spec_data["data_property_variations"]:
        print(f"  - {prop['property_name']}: {len(prop['property_values'])} values")

print()
print("Full spec YAML:")
print(spec.content_yaml)

In [ ]:
TARGET_PROPERTIES = [
    "Real Estate Loans > Construction & Development > 1-4 Family Construction concentration level (BANK value at 12/31/2025)",
    "Real Estate Loans > Multifamily concentration level (BANK value at 12/31/2025)",
]

for prop in spec_data.get("data_property_variations", []):
    if prop["property_name"] in TARGET_PROPERTIES:
        print(f"{prop['property_name']}")
        for v in prop["property_values"]:
            print(f"  {v}")
        print()

### Step 4: Edit the Spec — Override Distribution for 1-4 Family Construction

We create a new version of the spec by updating the `base_distributions` for the *1-4 Family Construction* property so that values **95** and **99** each carry **50 %** weight and all previously observed values carry **0 %**.

In [ ]:
import copy

CONSTRUCTION_PROP = (
    "Real Estate Loans > Construction & Development > 1-4 Family Construction"
    " concentration level (BANK value at 12/31/2025)"
)

CONSISTENCY_REQUIREMENT = """
Very important: the document must be 100% internally consistent. All generated numbers must be consistent with each other. You MUST use the calculator tool to verify every single numerical relationship in the document for every number.

Before using the calculator, first plan the complete structure of the document: every table, every row label, every column, every entity/location. Decide what line items exist and how they roll up into subtotals and totals. Only after that create SINGLE, COMPLETE script with ALL calculations included in that one script, using the calculator to verify/derive or compute every single number that will appear in the document. Specifically:
- This applies across ALL columns and ALL rows of ALL tables, everything must be verified in ONE script.
- You are not allowed to rely solely on mental arithmetic for any calculation. If a number is to appear in the document, the script must confirm or compute it.
- If the document contains multiple locations, entities, departments, or time periods, the calculator must compute numbers for ALL of them — not just one as a template.
- A couple of examples of required calculations: Every subtotal and total must be verified as the exact sum of its component line items. Every percentage must be verified as the correct division, rounded to the displayed precision. Every cross-entity consolidation (e.g. summing locations, departments, or facilities) must be verified for every row.

There is NO tolerance for even small errors, arithmetic operations, it's 0. This means if sum or % of whatever aggregate deviates from its correct value even by $0.01, this is a grave problem that can't be there after sample is generated.

If you really need to recompute some things even after the script execution, you should call the calculator again for those things. If you are already satisfied with execution results, do not call the calculator again, use what you got.
"""

updated_spec_data = copy.deepcopy(spec_data)

# Append consistency requirement to shared requirements
existing_requirements = updated_spec_data.get("requirements") or ""
updated_spec_data["requirements"] = existing_requirements + CONSISTENCY_REQUIREMENT

# Override distribution for 1-4 Family Construction
for prop in updated_spec_data["data_property_variations"]:
    if prop["property_name"] == CONSTRUCTION_PROP:
        for new_val in [95, 99]:
            if new_val not in prop["property_values"]:
                prop["property_values"].append(new_val)

        prop["base_distributions"] = {
            v: (50 if v in (95, 99) else 0)
            for v in prop["property_values"]
        }
        break

updated_yaml = yaml.dump({"spec": updated_spec_data}, allow_unicode=True, sort_keys=False)

updated_spec = df_client.dataframer.specs.update(spec_id=spec_id, content_yaml=updated_yaml)

print(f"New spec version created")
print(f"Spec ID: {updated_spec.id}")
print(f"Status:  {updated_spec.status}")

### Step 5: Create a Generation Run

Using the updated spec, we launch a run that generates **5 samples** with:
- **Revisions**: conformance only, max 1 cycle
- **Filtering**: conformance gate
- **Thinking budget**: 16 000 tokens
- **Tools**: calculator

In [ ]:
run = df_client.dataframer.runs.create(
    spec_id=updated_spec.id,
    number_of_samples=5,
    generation_model="anthropic/claude-opus-4-7",
    revision_types=["conformance"],
    max_revision_cycles=1,
    filtering_types=["conformance"],
    generation_thinking_budget=16000,
    tools=["calculator"],
    skip_outline=True,
)

run_id = run.id
print(f"Run started")
print(f"Run ID: {run_id}")


def run_not_ready(result):
    return result.status not in ("SUCCEEDED", "FAILED")


@retry(wait=wait_fixed(10), retry=retry_if_result(run_not_ready), stop=stop_never)
def poll_run_status(run_id):
    return df_client.dataframer.runs.retrieve(run_id=run_id)


print("Polling for run status...")
run_result = poll_run_status(run_id)

if run_result.status == "FAILED":
    raise RuntimeError("Run failed")

print(f"\nRun completed successfully!")
print(f"Run ID:            {run_id}")
print(f"Samples completed: {run_result.samples_completed}")
print(f"Samples failed:    {run_result.samples_failed}")

### Step 6: Run Cost & Download Samples

In [ ]:
import requests

# --- Cost ---
base_url = str(df_client.base_url).rstrip("/")
cost_data = requests.get(
    f"{base_url}/api/dataframer/runs/{run_id}/cost/",
    headers={"Authorization": f"Bearer {os.environ['DATAFRAMER_API_KEY']}"},
).json()

total_cost = cost_data.get("total_cost_cents")
print(f"Total Cost for all samples: ${total_cost / 100}")

# --- Download ---
def zip_not_ready(result):
    return result.download_url is None


@retry(wait=wait_fixed(5), retry=retry_if_result(zip_not_ready), stop=stop_never)
def poll_download_all(run_id):
    return df_client.dataframer.runs.files.download_all(run_id=run_id)


print("\nGenerating download ZIP...")
dl = poll_download_all(run_id)
print(f"ZIP ready — {dl.size_bytes:,} bytes")

output_dir = Path("output") / run_id
output_dir.mkdir(parents=True, exist_ok=True)

zip_data = requests.get(dl.download_url).content
with zipfile.ZipFile(io.BytesIO(zip_data)) as zf:
    zf.extractall(output_dir)
    names = sorted(zf.namelist())

print(f"\nExtracted {len(names)} files to {output_dir}/")
for name in names:
    print(f"  {name}")

### Step 7: Decide Next Steps: Human Review? Review Auto-generate Labels? More Elaborate Evaluations? or Fire off an evaluation of the generated datasets?

Potential next steps:
- add reviewer-confirmed labels for each PDF and question
- capture golden labels via human labels in the DataFramer app 
- compute per-bank, per-question, and per-document metrics
- compare multiple models, prompts, or extraction strategies on the same labeled set
- export reviewed labels for future regression testing
